In [9]:
import os
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = (
    "GPU-54aab195-984c-7336-8529-2469f43de4fd"
)

repo = Path("/home/v-jinyuning/LLMs-from-scratch")
work_dir = repo / "ch05/02_alternative_weight_loading"

# 使用之前准备好的 tokenizer 离线缓存
cache_dir = work_dir / "tokenizer_files/cache"
assert cache_dir.is_dir(), f"请检查 tokenizer 缓存路径：{cache_dir}"
os.environ["TIKTOKEN_CACHE_DIR"] = str(cache_dir)

sys.path.insert(0, str(repo / "ch05/01_main-chapter-code"))

import torch
import tiktoken
import torch.nn.functional as F

from previous_chapters import GPTModel, create_dataloader_v1
from gpt_train import train_model_simple, generate_and_print_sample

assert torch.cuda.is_available()
device = torch.device("cuda:0")
tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(32050104208)
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5090


In [10]:
data_path = repo / "ch02/01_main-chapter-code/the-verdict.txt"
assert data_path.is_file(), f"找不到语料文件：{data_path}"

text = data_path.read_text(encoding="utf-8")

# 与教材一致：按文本位置划分，前 90% 训练，后 10% 验证
split = int(0.9 * len(text))
train_text = text[:split]
val_text = text[split:]

batch_size = 2
context_length = 256

train_loader = create_dataloader_v1(
    train_text,
    batch_size=batch_size,
    max_length=context_length,
    stride=context_length,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

val_loader = create_dataloader_v1(
    val_text,
    batch_size=batch_size,
    max_length=context_length,
    stride=context_length,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

assert len(train_loader) > 0 and len(val_loader) > 0

print("文本开头：", text[:200])
print("训练 batch 数：", len(train_loader))
print("验证 batch 数：", len(val_loader))

文本开头： I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a
训练 batch 数： 9
验证 batch 数： 1


In [11]:
input_ids, target_ids = next(iter(train_loader))

print("输入 shape:", input_ids.shape)
print("标签 shape:", target_ids.shape)

print("第一条输入的前 12 个 ID：")
print(input_ids[0, :12].tolist())

print("第一条标签的前 12 个 ID：")
print(target_ids[0, :12].tolist())

# 标签确实是输入对应的原始序列向后移动一位
assert torch.equal(input_ids[:, 1:], target_ids[:, :-1])
print("右移一位检查通过")

print("\n输入文本片段：")
print(tokenizer.decode(input_ids[0, :40].tolist()))

print("\n对应目标文本片段：")
print(tokenizer.decode(target_ids[0, :40].tolist()))

输入 shape: torch.Size([2, 256])
标签 shape: torch.Size([2, 256])
第一条输入的前 12 个 ID：
[11, 17728, 257, 8500, 4417, 284, 670, 319, 438, 15464, 11, 355]
第一条标签的前 12 个 ID：
[17728, 257, 8500, 4417, 284, 670, 319, 438, 15464, 11, 355, 340]
右移一位检查通过

输入文本片段：
, presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity

对应目标文本片段：
 presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity.


In [12]:
config = {
    "vocab_size": 50257,
    "context_length": context_length,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

train_model = GPTModel(config).to(device)

optimizer = torch.optim.AdamW(
    train_model.parameters(),
    lr=5e-4,
    weight_decay=0.1,
)

print("模型设备:", next(train_model.parameters()).device)
print("实际参数量:", sum(p.numel() for p in train_model.parameters()))

模型设备: cuda:0
实际参数量: 162419712


In [ ]:
train_model.train()
optimizer.zero_grad(set_to_none=True)

# 数据也要搬到 GPU
inputs = input_ids.to(device)
targets = target_ids.to(device)

# 挑一个很小的参数张量，记录更新前的数值
tracked_param = train_model.final_norm.scale
before = tracked_param.detach().clone()

# 1. 前向
logits = train_model(inputs)
print("logits shape:", logits.shape)

# 2. 计算 loss
loss = F.cross_entropy(
    logits.flatten(0, 1),  # [2,256,50257] → [512,50257]
    targets.flatten(),    # [2,256]       → [512]
)
assert torch.isfinite(loss).item()
print("loss:", loss.item())

# 3. 反向
loss.backward()
assert tracked_param.grad is not None
assert torch.isfinite(tracked_param.grad).all().item()
print("被观察参数的梯度范数:", tracked_param.grad.norm().item())

# 4. 更新
optimizer.step()

change = (tracked_param.detach() - before).abs().max().item()
print("该参数的最大变化量:", change)

# 释放这次检查不再需要的张量和梯度
optimizer.zero_grad(set_to_none=True)
del logits, loss, before

logits shape: torch.Size([2, 256, 50257])
loss: 11.024642944335938
被观察参数的梯度范数: 0.031124429777264595
该参数的最大变化量: 0.0005500316619873047


In [ ]:
train_losses, val_losses, tokens_seen = train_model_simple(
    model=train_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=2,
    eval_freq=5,
    eval_iter=2,
    start_context="Every effort moves you",
    tokenizer=tokenizer,
)

Ep 1 (Step 000000): Train loss 9.152, Val loss 9.495
Ep 1 (Step 000005): Train loss 7.397, Val loss 7.661
Every effort moves you                                                  
Ep 2 (Step 000010): Train loss 6.381, Val loss 6.731
Ep 2 (Step 000015): Train loss 5.786, Val loss 6.590
Every effort moves you SE SE Size SE Size SE SE SE SE SE▬ifted Nikeuntu cooledresources SE▬untu▬untu▬untu SEuntu▬untu SEuntu▬ SEuntuuntu▬untu▬untu▬ vend SEuntu▬untu▬untu▬untu cooleduntu▬


In [15]:
checkpoint_path = work_dir / "gpt2_toy_training_checkpoint.pt"

torch.save(
    {
        "config": config,
        "model_state_dict": train_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
    },
    checkpoint_path,
)

print("已保存到:", checkpoint_path)

已保存到: /home/v-jinyuning/LLMs-from-scratch/ch05/02_alternative_weight_loading/gpt2_toy_training_checkpoint.pt


In [35]:
####### 手写一遍单batch ######

train_model.train()
optimizer.zero_grad(set_to_none=True)

inputs = input_ids.to(device)
targets = target_ids.to(device)
print(f'inputs.shape={inputs.shape}\ntargets.shape={targets.shape}')

logits = train_model(inputs)
print(f'logits.shape={logits.shape}')

loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
loss.backward()

optimizer.step()

####### 手写单epoch #######

train_model.train()
optimizer.zero_grad(set_to_none=True)

train_losses = []
for input_ids, target_ids in train_loader:
    
    inputs = input_ids.to(device)
    targets = target_ids.to(device)
    # print(f'inputs.shape={inputs.shape}\ntargets.shape={targets.shape}')

    logits = train_model(inputs)
    # print(f'logits.shape={logits.shape}')

    loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
    train_losses.append(loss.item())
    loss.backward()

    optimizer.step()

print(f'训练损失：{train_losses}')
print(sum(train_losses) / len(train_losses))

inputs.shape=torch.Size([2, 256])
targets.shape=torch.Size([2, 256])
logits.shape=torch.Size([2, 256, 50257])
训练损失：[1.1861422061920166, 1.2272727489471436, 1.0322362184524536, 1.1073260307312012, 1.2564908266067505, 0.9918918609619141, 1.222987174987793, 1.3857195377349854, 1.321221113204956]
1.1923653019799128


In [36]:
##### val验证 ######

train_model.eval()
val_losses = []

with torch.no_grad():
    for input_ids, target_ids in val_loader:

        inputs = input_ids.to(device)
        targets = target_ids.to(device)
        # print(f'inputs.shape={inputs.shape}\ntargets.shape={targets.shape}')

        logits = train_model(inputs)
        # print(f'logits.shape={logits.shape}')

        loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        val_losses.append(loss.item())

print(f'验证损失：{val_losses}')

验证损失：[4.7295989990234375]


In [37]:
checkpoint_path = work_dir / "gpt2_toy_manual_state.pt"
torch.save(train_model.state_dict(), checkpoint_path)

In [39]:
####### 重现模型和参数 #######
restored_model = GPTModel(config).to(device)
saved_state = torch.load(checkpoint_path, weights_only=True)
restored_model.load_state_dict(saved_state)

<All keys matched successfully>

In [40]:
restored_model.eval()
train_model.eval()

sample_ids, _ = next(iter(val_loader))
samples = sample_ids.to(device)

with torch.no_grad():
    original_logits = train_model(samples)
    new_logits = restored_model(samples)

print(torch.allclose(original_logits, new_logits))

True


In [41]:
p = sum([para.numel() for para in restored_model.parameters()])
print(p)

para_size = sum([para.numel() * para.element_size() for para in restored_model.parameters()])
print(para_size)

162419712
649678848
